In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.nn.utils.parametrizations import weight_norm
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
class OutputCrop1d(nn.Module):
    def __init__(self, crop_size: int):
        super().__init__()
        self.crop_size = crop_size

    def forward(self, x: torch.Tensor):
        return x[:, :, :-self.crop_size].contiguous()


class TemporalConvUnit(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        dilation: int,
        stride: int = 1,
        dropout: float = 0.2,
        name: str | None = None
    ):
        super().__init__()
        self.name = name
        
        # Weight normalisation: https://arxiv.org/abs/1602.07868
        self.conv = weight_norm(
            nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                padding=padding,
                dilation=dilation,
                stride=stride,
            )
        )
        self.conv.weight.data.normal_(0, 0.01)
        self.crop = OutputCrop1d(padding)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv, self.crop, self.relu, self.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)
    

class TemporalConvBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        padding: int,
        dilation: int,
        stride: int = 1,
        dropout: float = 0.2,
    ):
        """
        :param in_channels: Number of input channels. Equivalent to number of features in the time series at each timestep
        :param out_channels: Number of output channels. Represents the number of feature maps to learn
        :param kernel_size: Number of weights per filter.
        :param padding: Size of padding to apply to both sides of the input

        """
        super().__init__()
        
        self.unit1 = TemporalConvUnit(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
            stride=stride,
            dropout=dropout,
        )
        self.unit2 = TemporalConvUnit(
            in_channels=out_channels,
            out_channels=out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=padding,
            stride=stride,
            dropout=dropout,
        )
        self.net = nn.Sequential(self.unit1, self.unit2)

        # Residual connection
        if in_channels != out_channels:
            self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=1)
            self.conv.weight.data.normal_(0, 0.01)
        else:
            self.conv = None
        
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.net(x)
        res = x if self.conv is None else self.conv(x)
        return self.relu(out + res)
    

class TemporalConvNetDecoder(nn.Module):
    def __init__(self, in_features: int, out_features: int = 1):
        super().__init__()
        self.conv = nn.Conv1d(in_features, out_features, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class TemporalConvNet(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        kernel_size: int = 2,
        dropout: float = 0.2
    ):
        super().__init__()
        
        num_encoded_features = 5
        dilation = 1
        self.encoder = TemporalConvBlock(
            in_channels=in_features,
            out_channels=num_encoded_features,
            kernel_size=kernel_size,
            dilation=dilation,
            padding=dilation * (kernel_size - 1),
            dropout=dropout,
        )
        self.decoder = TemporalConvNetDecoder(
            in_features=num_encoded_features,
            out_features=out_features,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_encoded = self.encoder(x)
        return self.decoder(x_encoded)

In [ ]:
# Can TCB already learn something?

n_timesteps = 1000
period = 24
timesteps = np.arange(n_timesteps)
timeseries = np.sin(2 * np.pi * timesteps / period)

plt.plot(timesteps, timeseries)

In [ ]:
# Prepare tensor dataset
in_seq_length = out_seg_length = 25
horizon = 10

Xs, ys = [], []
for idx in range(0, len(timeseries) - in_seq_length - horizon + 1):
    X_start, X_end = idx, idx + in_seq_length
    y_start, y_end = X_start + horizon, X_end + horizon
    Xs.append(timeseries[X_start:X_end])
    ys.append(timeseries[y_start:y_end])

Xs = torch.tensor(np.array(Xs), dtype=torch.float).view(-1, 1, in_seq_length)
ys = torch.tensor(np.array(ys), dtype=torch.float).view(-1, 1, out_seg_length)

ds = TensorDataset(Xs, ys)
dl = DataLoader(ds, batch_size=32)

In [ ]:
model = TemporalConvNet(
    in_features=1,
    out_features=1,
    kernel_size=30,
    dropout=0.2
)
loss_fn = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-03)

model.train()
epoch_loss, batch_loss = [], []
n_epochs = 100
pgbar = tqdm(range(n_epochs))
for epoch in pgbar:
    for X, y in dl:
        optimizer.zero_grad()
        
        y_hat = model(X)
        loss = loss_fn(y, y_hat)
        loss.backward()
        optimizer.step()
        
        loss_detach = float(loss.detach())
        batch_loss.append(loss_detach)
    
    epoch_loss.append(loss_detach)
    pgbar.set_description(f"Epoch [{epoch + 1} / {n_epochs}] - Loss = {loss_detach:.3f}")

In [ ]:
X_test, y_test = ds[16]
X_test = X_test.view(-1, 1, in_seq_length)

model.eval()
y_hat = model(X_test)

fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(np.arange(in_seq_length), X_test.squeeze().numpy())
ax[0].axvline(horizon)

ax[1].plot(np.arange(horizon, in_seq_length + horizon), y_test.squeeze().numpy())
ax[1].plot(np.arange(horizon, in_seq_length + horizon), y_hat.detach().squeeze().numpy())
ax[1].axvline(horizon)